[Setup instructions](./Summary.md#topic-notebooks)


In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
import sys

_shared_dir = Path.cwd()
if not (_shared_dir / "common.py").is_file():
    _shared_dir = _shared_dir / "02-regression"
if not (_shared_dir / "common.py").is_file():
    raise FileNotFoundError(
        "Run this notebook from the machine-learning or 02-regression folder."
    )
if str(_shared_dir) not in sys.path:
    sys.path.insert(0, str(_shared_dir))

from common import (
    df_train,
    df_val,
    df_test,
    y_train,
    y_val,
    y_test,
    prepare_X,
    train_linear_regression_reg,
    rmse,
)


# Using the model

Now that we have tuned the regularization parameter, we can train the final model on the combined training+validation data and then evaluate it on the untouched test set.

The idea is simple:
- train on as much data as possible
- keep the test set only for the final evaluation
- use the validation set only for model selection

This gives a more realistic estimate of how the model will perform in production.


In [366]:
df_full_train = pd.concat([df_train, df_val])
df_full_train = df_full_train.reset_index(drop=True)

X_full_train = prepare_X(df_full_train)
y_full_train = np.concatenate([y_train, y_val])

w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)


In [367]:
X_test = prepare_X(df_test)
y_pred = w0 + X_test.dot(w)
score = rmse(y_test, y_pred)
score


np.float64(0.4517749306915374)

# Predicting the price of a car

We can use the trained model just like it would be used in practice: take a car description, build the feature matrix, and get a predicted price.

In this example, we take a car from the test set and pretend it is a new car.


In [368]:
car = df_test.iloc[20].to_dict()
car


{'make': 'toyota',
 'model': 'sienna',
 'year': 2015,
 'engine_fuel_type': 'regular_unleaded',
 'engine_hp': 266.0,
 'engine_cylinders': 6.0,
 'transmission_type': 'automatic',
 'driven_wheels': 'front_wheel_drive',
 'number_of_doors': 4.0,
 'market_category': nan,
 'vehicle_size': 'large',
 'vehicle_style': 'passenger_minivan',
 'highway_mpg': 25,
 'city_mpg': 18,
 'popularity': 2031}

In [369]:
df_small = pd.DataFrame([car])
X_small = prepare_X(df_small)

y_pred = w0 + X_small.dot(w)
y_pred = y_pred[0]
y_pred


np.float64(10.462651700050063)

In [370]:
np.expm1(y_pred)


np.float64(34983.19615434434)

In [371]:
np.expm1(y_test[20])


np.float64(35000.00000000001)